# Caso 5: localización de depósitos con múltiples objetivos

---


## Instrucciones generales

El primer paso antes de resolver este laboratorio es leer y entender el **enunciado del caso**.

Este laboratorio tiene las siguientes secciones:
* **Formulación**: en este caso particular, definimos dos funciones objetivo $z_1$ y $z_2$
* **Importación de librerías**
* **Creación de parámetros**
* **Modelado**: en esta práctica, haremos tres (3) implementaciones del mismo problema:
    * **Minimización de costos**
    * **Maximización de la satisfacción**
    * **Maximización de la satisfacción con restricción de costos**
* **Reporte de Resultados**

Este tipo de actividades se evaluará sobre un total de 100 puntos. Las celdas calificables se distinguen por tener la instrucción `# your code here`. Antes de estas celdas encontrarás instrucciones y consejos para resolver las preguntas, también el puntaje que le corresponde.

¡Éxitos!

## Formulación
---

Te presentamos la formulación del caso de la semana de forma resumida. Te recomendamos revisar la formulación una vez hayas leído el enunciado del caso. Es bueno que te familiarices con los elementos de la formulación antes de iniciar la implementación.

### Conjuntos y Parámetros
>#### **Conjuntos**
>* $I:$ Depósitos
>* $J:$ Centros de acopio consolidados (CACs)

>#### **Parámetros**
>* $k_i:$ Capacidad del depósito $i \in I$ (miles de toneladas por año)
>* $f_i:$ Costo de operación del depósito $i \in I$ (millones de pesos por año)
>* $d_j:$ Producción proyectada del CAC $j\in J$ (miles de tonaledas por año)
>* $q:$ Costo anualizado (por cada mil toneladas por kilómetro) de transportar café
>* $r:$ Distancia máxima (en kilómetros) entre el CAC y el depósito asignado para estar "bien" atendido
>* $h_{ij}:$ Distancia (en kilómetros) entre el CAC $j \in J$ y el depósito $i \in I$
>* $c_{ij}:$ Costo anualizado de atender el CAC $j\in J$ con el depósito $i\in I$ (Se calcula como: $c_{ij} = q\cdot d_j \cdot h_{ij}$)

### Variables de Decisión
>* $x_{ij}=\begin{cases}1, & \text{si el CAC } j \in J \text{ es atendido por el depósito } i \in I \\0, & \text{de lo contrario} \end{cases}$
>* $y_{i}=\begin{cases} 1, & \text{si se decide operar el depósito } i \in I  \\ 0, & \text{de lo contrario} \end{cases}$
    
### Restricciones
>1. Cada CAC debe ser atendido por un único depósito
>>`# Para desarrollo del estudiante`
>2. No se debe superar la capacidad de los depósitos y sólo se puede atender CACs desde un depósito si se decide operar el mismo.
>>$\sum_{j \in J}d_{j}x_{ij} \leq k_{i}y_{i}, \; \forall i \in I$

> **Naturaleza de variables**
>>$x_{ij} \in \{0,1\} , \;\forall i \in I, j \in J$
>>
>>$y_{i} \in \{0,1\} , \;\forall i \in I$

### Función Objetivo
>* Minimizar los costos totales de operación y transporte
>>`# Para desarrollo del estudiante`
>* Maximizar satisfacción de los CACs
>>$\max z_2 = \sum_{j \in J} d_{j} \sum_{\{i \in I | h_{ij} \leq r\}} x_{ij}$

## Importación de librerías
---
En esta práctica usaremos:
* El paquete `pandas` es muy útil para el análisis de datos en general. Le asignamos el alias de `pd`.
* El paquete `pulp` permite crear modelos de optimización, crear variables, añadir restricciones y muchos más. Le asignamos el alias de `lp`.
* La función `distance` del módulo `geopy.distance` nos permite hallar fácilmente la distancia geodéisca en kilómetros entre dos pares de coordenadas de longitud y latitud.


In [72]:
import pandas as pd
import pulp as lp
from geopy.distance import distance

## Creación de Parámetros
---

### Lectura del archivo de soporte

Los datos que necesitamos para esta práctica se encuentran disponibles en el archivo `Soporte Caso 5.xlsx`.
En este archivo encontraremos los mismo datos del enunciado.
Importamos las hojas `CACs` y `Depositos` del archivo `Soporte Caso 5.xlsx`.
Estas hojas son importadas como objetos `DataFrame` de `pandas`.

In [73]:
cacs = pd.read_excel('Soporte Caso 5.xlsx', sheet_name='CACs')
depositos = pd.read_excel('Soporte Caso 5.xlsx', sheet_name='Depositos')

### Procesamiento de archivos de soporte

En este paso, se crean los **Conjuntos** y **Parámetros**.
Es necesario dejar todo expresado en términos de listas y diccionarios para facilitar la implementación del modelo en PuLP. Adicionalmente, debemos procesar las coordenadas de longitud y latitud para obtener las distancias entre CACs y Depósitos.

In [74]:
I = depositos.Municipio.to_list()
J = cacs.Municipio.to_list()

capacidad = {row["Municipio"]: row["Capacidad"] for _, row in depositos.iterrows()}
costo_fijo = {row["Municipio"]: row["CostoFijo"] for _, row in depositos.iterrows()}
depositos_lat_lon = {
    row["Municipio"]: (row["Latitud"], row["Longitud"])
    for _, row in depositos.iterrows()
}

produccion = {row["Municipio"]: row["Produccion"] for _, row in cacs.iterrows()}
cacs_lat_lon = {
    row["Municipio"]: (row["Latitud"], row["Longitud"]) for _, row in cacs.iterrows()
}

q = 90  # Pesos anualizados por cada mil toneladas de café por kilómetro
r = 125  # Kilómetros

distancia = {
    (i, j): distance(depositos_lat_lon[i], cacs_lat_lon[j]).kilometers
    for i in I
    for j in J
}

In [75]:
distancia

{('Medellín, Antioquia', 'Andes, Antioquia'): 81.6407744153149,
 ('Medellín, Antioquia', 'Medellín, Antioquia'): 0.0,
 ('Medellín, Antioquia', 'Dabeiba, Antioquia'): 107.79278431038877,
 ('Medellín, Antioquia', 'Salgar, Antioquia'): 53.335670657757895,
 ('Medellín, Antioquia', 'San Pablo de Borbur, Boyacá'): 177.7022202999153,
 ('Medellín, Antioquia', 'Labranzagrande, Boyacá'): 341.843300383342,
 ('Medellín, Antioquia', 'Miraflores, Boyacá'): 295.50792681396376,
 ('Medellín, Antioquia', 'Moniquirá, Boyacá'): 230.85325502996707,
 ('Medellín, Antioquia', 'Manizales, Caldas'): 132.50832910815635,
 ('Medellín, Antioquia', 'Anserma, Caldas'): 119.36408078562327,
 ('Medellín, Antioquia', 'Pensilvania, Caldas'): 106.38936050657013,
 ('Medellín, Antioquia', 'Riosucio, Caldas'): 91.7929643629694,
 ('Medellín, Antioquia', 'Aguadas, Caldas'): 77.80655349448206,
 ('Medellín, Antioquia', 'Morales, Cauca'): 399.20320142759863,
 ('Medellín, Antioquia', 'El Tambo, Cauca'): 442.9927758625815,
 ('Medell

**Pregunta 1 (10 puntos)**

* Crea el parámetro de costo de transporte $c_{ij}$ en un diccionario llamado `costo_transporte`
* Las **llaves** de este diccionario deben ser los pares $(i,j)$, es decir, (depósitos, CACs)
* Los **valores** de este diccionario deben ser los costos de transporte definidos en la formulación

### Fórmula y explicación en palabras — Pregunta 1

**Fórmula:**  
\[
c_{ij} = q \cdot d_j \cdot h_{ij}
\]  
(\(d_j\): producción del CAC \(j\); \(h_{ij}\): distancia en km; \(q\): parámetro del enunciado.)

**Qué haces en la casilla:** Armas un diccionario `costo_transporte` con clave `(i, j)` y valor \(q \times\) `produccion[j]` \(\times\) `distancia[(i,j)]` para todos los depósitos y CACs.

**Por qué:** El modelo lineal necesita coeficientes numéricos \(c_{ij}\) listos para multiplicar por \(x_{ij}\) en la función objetivo de costos.

**Cómo decirlo en voz:** *“Para cada posible ruta entre un depósito y un CAC calculo cuánto cuesta anualmente transportar allí toda la producción de ese CAC, usando la tarifa, la producción y la distancia.”*


In [76]:
costo_transporte = {
    (i, j): q * produccion[j] * distancia[(i, j)]
    for i in I
    for j in J
}


In [77]:
# Esta celda esta reservada para uso del equipo docente

In [78]:
# Esta celda esta reservada para uso del equipo docente

**Celda de Prueba (0 puntos)**

Es una buena práctica imprimir algunos objetos que contienen los parámetros en la consola luego de crearlos. De esta forma puedes corregir errores y familiarizarte con las estructuras de datos que se van a utilizar. Puedes hacer estas pruebas en la celda a continuación.

* **Esta celda no es calificable**

### Fórmula y explicación en palabras — Celda de prueba (no calificada)

**Fórmula:** No aplica; es depuración.

**Qué haces en la casilla:** Imprimes una muestra de `costo_transporte` y cuántos pares \((i,j)\) tienes.

**Por qué:** Detectar errores temprano (valores absurdos, claves mal formadas) antes de armar el modelo grande.

**Cómo decirlo en voz:** *“Reviso que los costos de transporte se calcularon bien y que tengo un valor por cada par depósito–CAC.”*


In [79]:
# Celda de prueba (no calificada): inspección opcional
print("Muestra costo_transporte:", list(costo_transporte.items())[:2])
print("Pares (i,j):", len(costo_transporte))


Muestra costo_transporte: [(('Medellín, Antioquia', 'Andes, Antioquia'), 259887.0771962719), (('Medellín, Antioquia', 'Medellín, Antioquia'), 0.0)]
Pares (i,j): 1045


## Modelado - Minimización de costos ($z_1$)
---

### Declaración del modelo

In [80]:
problema = lp.LpProblem(sense=lp.LpMinimize)

### Variables de Decisión

>* $x_{ij}=\begin{cases}1, & \text{si el CAC } j \in J \text{ es atendido por el depósito } i \in I \\0, & \text{de lo contrario} \end{cases}$
>* $y_{i}=\begin{cases} 1, & \text{si se decide operar el depósito } i \in I  \\ 0, & \text{de lo contrario} \end{cases}$

In [81]:
x = lp.LpVariable.dicts("atender", [(i, j) for i in I for j in J], lowBound = 0, cat=lp.LpBinary)
y = lp.LpVariable.dicts("operar", I, lowBound = 0, cat=lp.LpBinary)

### Función Objetivo
Minimizar los costos totales de operación y transporte
>`# Para desarrollo del estudiante`

**Pregunta 2 (10 puntos)**
* Crea la función objetivo y agrégala al modelo `problema`

> **Ejemplo**:
>> $ \sum_{i \in I}c_i x_i$
es equivalente a `lp.lpSum(c[i]*x[i] for i in I)`

### Fórmula y explicación en palabras — Pregunta 2

**Fórmula:**  
\[
\min z_1 = \sum_{i \in I} f_i\, y_i + \sum_{i \in I}\sum_{j \in J} c_{ij}\, x_{ij}
\]

**Qué haces en la casilla:** Construyes la función objetivo: suma de costos fijos por depósitos abiertos más suma de costos de transporte según las asignaciones \(x_{ij}\), y la asignas al objeto `problema` con `+=`.

**Por qué:** Es la traducción directa del criterio económico del caso; PuLP buscará la asignación y apertura que minimice ese total.

**Cómo decirlo en voz:** *“Minimizo lo que gasto en abrir depósitos y en mover el café desde cada depósito a cada CAC.”*


In [82]:
problema += lp.lpSum(costo_fijo[i] * y[i] for i in I) + lp.lpSum(
    costo_transporte[i, j] * x[i, j] for i in I for j in J
)


In [83]:
# Esta celda esta reservada para uso del equipo docente

In [84]:
# Esta celda esta reservada para uso del equipo docente

### Restricciones

____
**Ejemplo**
> La siguiente restricción: $\sum_{i \in I} a_{ij} x_{ij} \geq 1, \; \forall j \in J$ es equivalente a:
>    * `for j in J:`
>        * `model += lp.lpSum(a[i,j]*x[i,j] for i in I) >= 1, 'R1_'+str(j)`
    
**Advertencia**: En `pulp` no es recomendable sobreescribir restricciones, entonces, si ya creaste una restricción y quieres crearla de nuevo para corregir algo, asegúrate de volver a crear el modelo `problema` desde el principio. (Nosotros haremos esto antes de calificar, no te preocupes)

**Pregunta 3 (10 puntos)**

* Crea la siguiente restricción, asígnale el nombre `'R1_'+str(<indice_del_para_todo>)` y añádela al modelo:
>1. Cada CAC debe ser atendido por un único depósito
>>`# Para desarrollo del estudiante`

### Fórmula y explicación en palabras — Pregunta 3 (R1)

**Fórmula:**  
\[
\sum_{i \in I} x_{ij} = 1 \qquad \forall j \in J
\]

**Qué haces en la casilla:** Para cada CAC \(j\), la suma de \(x_{ij}\) sobre todos los depósitos \(i\) debe ser exactamente 1. En PuLP das nombre `'R1_'+str(j)`.

**Por qué:** Garantiza que todo centro quede atendido y por un solo depósito (no fraccionado entre varios en este modelo).

**Cómo decirlo en voz:** *“Cada centro de acopio elige un único depósito que lo surta.”*


In [85]:
for j in J:
    problema += lp.lpSum(x[i, j] for i in I) == 1, "R1_" + str(j)


In [86]:
# Esta celda esta reservada para uso del equipo docente

**Pregunta 4 (5 puntos)**

* Crea la siguiente restricción, asígnale el nombre `'R2_'+str(<indice_del_para_todo>)` y añádela al modelo:
>2. No se debe superar la capacidad de los depósitos y sólo se puede atender CACs desde un depósito si se decide operar el mismo.
>>$\sum_{j \in J}d_{j}x_{ij} \leq k_{i}y_{i}, \; \forall i \in I$

### Fórmula y explicación en palabras — Pregunta 4 (R2)

**Fórmula:**  
\[
\sum_{j \in J} d_j\, x_{ij} \;\leq\; k_i\, y_i \qquad \forall i \in I
\]

**Qué haces en la casilla:** Para cada depósito \(i\), la producción total asignada no supera \(k_i\) si está abierto (\(y_i=1\)); si está cerrado, el lado derecho es 0 y no puede recibir asignaciones.

**Por qué:** Modela capacidad física y coherencia lógica entre variables \(x\) e \(y\).

**Cómo decirlo en voz:** *“No puedo cargar más del límite del depósito, y no puedo usar un depósito que no abrí.”*


In [87]:
for i in I:
    problema += (
        lp.lpSum(produccion[j] * x[i, j] for j in J) <= capacidad[i] * y[i]
    ), "R2_" + str(i)


In [88]:
# Esta celda esta reservada para uso del equipo docente

In [89]:
# Esta celda esta reservada para uso del equipo docente

### Invocar el optimizador

In [90]:
print(lp.LpStatus[problema.solve()])

Optimal


## Reporte de resultados - Minimización de costos ($z_1$)
---

**Función objetivo $z_1$**

In [91]:
z1 = lp.value(problema.objective)
min_costo = z1
print(f"Costo Total: ${z1: .2f}")
print(f"Costo Total Relativo al Mínimo Costo: {z1/min_costo*100: .2f}%")

Costo Total: $ 4318336.74
Costo Total Relativo al Mínimo Costo:  100.00%


**Función objetivo $z_2$**

**Pregunta 5 (5 puntos)**

* Guarda en una variable `z2` el valor de la satisfacción total dado por la expresión:
> $z_2 = \sum_{j \in J} d_{j} \sum_{\{i \in I | h_{ij} \leq r\}} x_{ij}$

**Recuerda que** en PuLP puedes usar la función `lp.value(<expresion>)` para evaluar una expresión, reemplazando los valores de las variables por aquellos de la solución óptima. Esta función sólo debe ser llamada luego de usar `<modelo>.solve()` y haber obtenido una solución óptima.

### Fórmula y explicación en palabras — Pregunta 5

**Fórmula:**  
\[
z_2 = \sum_{j \in J} d_j \sum_{\{i \in I \mid h_{ij} \leq r\}} x_{ij}
\]

**Qué haces en la casilla:** Después de minimizar costos, evalúas esa suma con la solución óptima (`lp.value` sobre un `lp.lpSum` que filtra `distancia[(i,j)] <= r`).

**Por qué:** El primer modelo optimiza \(z_1\), no \(z_2\). Esta casilla **mide** cuánta satisfacción obtuviste “de regalo” o a costa de rutas largas.

**Cómo decirlo en voz:** *“Con la solución más barata, calculo cuánta producción queda dentro del umbral de distancia que define buen servicio.”*


In [92]:
z2 = lp.value(
    lp.lpSum(
        produccion[j] * x[i, j]
        for j in J
        for i in I
        if distancia[(i, j)] <= r
    )
)


In [93]:
# Esta celda esta reservada para uso del equipo docente

**Depósitos en operación**

In [94]:
print("Se decidió operar", sum(y[i].value() for i in I), "depósitos")

Se decidió operar 15.0 depósitos


**Asignación de CACs a Depósitos**

In [95]:
matrix = []
for j in J:
    row = []
    for i in I:
        if y[i].value() == 1:
            if x[i,j].value() == 1:
                row.append('X')
            elif x[i,j].value() == 0:
                row.append('-')
            else:
                row.append('Error')
    matrix.append(row)

df = pd.DataFrame(matrix, index=J, columns=[i for i in I if y[i].value() == 1])
df.head(10)

,"Medellín, Antioquia","La Dorada, Caldas","Aguadas, Caldas","Salamina, Caldas","Popayán, Cauca","Valledupar, Cesar","Santana, Huila","Neiva, Huila","Cúcuta, Nor. de Santander","Pasto, Nariño","Génova, Quindío","Filandia, Quindío","Bucaramanga, Santander","Barbosa, Santander","Cali, Valle del Cauca"
"Andes, Antioquia",X,-,-,-,-,-,-,-,-,-,-,-,-,-,-
"Medellín, Antioquia",X,-,-,-,-,-,-,-,-,-,-,-,-,-,-
"Dabeiba, Antioquia",X,-,-,-,-,-,-,-,-,-,-,-,-,-,-
"Salgar, Antioquia",X,-,-,-,-,-,-,-,-,-,-,-,-,-,-
"San Pablo de Borbur, Boyacá",-,X,-,-,-,-,-,-,-,-,-,-,-,-,-
"Labranzagrande, Boyacá",-,-,-,-,-,-,-,-,-,-,-,-,X,-,-
"Miraflores, Boyacá",-,-,-,-,-,-,-,-,-,-,-,-,-,X,-
"Moniquirá, Boyacá",-,-,-,-,-,-,-,-,-,-,-,-,-,X,-
"Manizales, Caldas",-,-,-,X,-,-,-,-,-,-,-,-,-,-,-
"Anserma, Caldas",-,-,X,-,-,-,-,-,-,-,-,-,-,-,-


### Visualizaciones
---

**Mapa de la asignación**

In [96]:
# Para los mapas
import folium
# Para los marcadores de los mapas
from folium.plugins import BeautifyIcon

m = folium.Map(location=[6.2, -74.5], tiles="OpenStreetMap", zoom_start=6)

for j, lat_lon in cacs_lat_lon.items():
    folium.Marker(
        location=lat_lon,
        tooltip=j,
        icon=BeautifyIcon(
            icon="circle",
            inner_icon_style="color:blue;font-size:7px;opacity:0.9;position: relative;top:-0.5px;",
            background_color="transparent",
            border_color="transparent",
        ),
    ).add_to(m)
for i, lat_lon in depositos_lat_lon.items():
    if y[i].value() > 0:
        folium.Marker(
            location=lat_lon,
            tooltip=i,
            icon=BeautifyIcon(
                icon="caret-up",
                inner_icon_style="color:red;font-size:20px;opacity:0.9;position: relative;top:-4.5px;",
                background_color="transparent",
                border_color="transparent",
            ),
        ).add_to(m)

red = [(i, j) for i in I for j in J if x[i, j].value() > 0]
for i, j in red:
    folium.PolyLine(
        [depositos_lat_lon[i], cacs_lat_lon[j]], color="black", weight=1, opacity=1
    ).add_to(m)

m

## Modelado - Maximización de satisfacción ($z_2$)
---
A continuación queremos explorar el cambio en las funciones objetivo $z_1$ y $z_2$ cuando se prioriza $z_2$. Las restricciones y variables del problema permanecen igual, pero la solución cambiará.

### Declaración del modelo

In [97]:
problema = lp.LpProblem(sense=lp.LpMaximize)

### Variables de Decisión

>* $x_{ij}=\begin{cases}1, & \text{si el CAC } j \in J \text{ es atendido por el depósito } i \in I \\0, & \text{de lo contrario} \end{cases}$
>* $y_{i}=\begin{cases} 1, & \text{si se decide operar el depósito } i \in I  \\ 0, & \text{de lo contrario} \end{cases}$

In [98]:
x = lp.LpVariable.dicts('atender', [(i,j) for i in I for j in J], cat=lp.LpBinary)
y = lp.LpVariable.dicts('operar', I, cat=lp.LpBinary)

### Función Objetivo
Maximizar satisfacción de los CACs
>$\max z_2 = \sum_{j \in J} d_{j} \sum_{\{i \in I | h_{ij} \leq r\}} x_{ij}.$

Esta expresión es equivalente a:
>$\max z_2 = \sum_{j \in J} \sum_{\{i \in I | h_{ij} \leq r\}}d_{j} x_{ij}.$

**Pregunta 6 (5 puntos)**
* Crea la función objetivo y agrégala al modelo `problema`

> **Ejemplo**:
>> $ \sum_{i \in I}c_i x_i$
es equivalente a `lp.lpSum(c[i]*x[i] for i in I)`

### Fórmula y explicación en palabras — Pregunta 6

**Fórmula:**  
\[
\max z_2 = \sum_{j \in J}\;\sum_{\{i \in I \mid h_{ij} \leq r\}} d_j\, x_{ij}
\]

**Qué haces en la casilla:** Sumas `produccion[j] * x[i,j]` solo cuando la distancia \(h_{ij}\) es menor o igual que \(r\) (125 km), y lo pones como objetivo del `problema`.

**Por qué:** Solo cuenta como “satisfacción” la producción asignada a un depósito **suficientemente cercano**; así el solver premia cobertura de calidad, no solo cualquier asignación barata.

**Cómo decirlo en voz:** *“Quiero maximizar las toneladas que quedan atendidas dentro del radio de buen servicio.”*


In [99]:
problema += lp.lpSum(
    produccion[j] * x[i, j]
    for j in J
    for i in I
    if distancia[(i, j)] <= r
)


In [100]:
# Esta celda esta reservada para uso del equipo docente

In [101]:
# Esta celda esta reservada para uso del equipo docente

### Restricciones
____

**Pregunta 7 (10 puntos)**

* Crea la siguiente restricción, asígnale el nombre `'R1_'+str(<indice_del_para_todo>)` y añádela al modelo:
>1. Cada CAC debe ser atendido por un único depósito
>>`# Para desarrollo del estudiante`

### Fórmula y explicación en palabras — Pregunta 7 (R1, segundo modelo)

**Fórmula:**  
\[
\sum_{i \in I} x_{ij} = 1 \qquad \forall j \in J
\]

**Qué haces en la casilla:** Igual que la Pregunta 3: asignación única por CAC.

**Por qué:** El segundo modelo solo cambia la función objetivo; la factibilidad del problema es la misma.

**Cómo decirlo en voz:** *“Cada CAC tiene exactamente un depósito asignado.”*


In [102]:
for j in J:
    problema += lp.lpSum(x[i, j] for i in I) == 1, "R1_" + str(j)


In [103]:
# Esta celda esta reservada para uso del equipo docente

**Pregunta 8 (5 puntos)**

* Crea la siguiente restricción, asígnale el nombre `'R2_'+str(<indice_del_para_todo>)` y añádela al modelo:
>2. No se debe superar la capacidad de los depósitos y sólo se puede atender CACs desde un depósito si se decide operar el mismo.
>>$\sum_{j \in J}d_{j}x_{ij} \leq k_{i}y_{i}, \; \forall i \in I$

### Fórmula y explicación en palabras — Pregunta 8 (R2, segundo modelo)

**Fórmula:**  
\[
\sum_{j \in J} d_j\, x_{ij} \;\leq\; k_i\, y_i \qquad \forall i \in I
\]

**Qué haces en la casilla:** Misma implementación que en la Pregunta 4: capacidad y vínculo con apertura de depósito.

**Por qué:** Al cambiar el objetivo a maximizar \(z_2\), las reglas operativas siguen siendo las mismas.

**Cómo decirlo en voz:** *“Respeto capacidad y que solo se use un depósito si está abierto.”*


In [104]:
for i in I:
    problema += (
        lp.lpSum(produccion[j] * x[i, j] for j in J) <= capacidad[i] * y[i]
    ), "R2_" + str(i)


In [105]:
# Esta celda esta reservada para uso del equipo docente

In [106]:
# Esta celda esta reservada para uso del equipo docente

### Invocar el optimizador

In [107]:
print(lp.LpStatus[problema.solve()])

Optimal


## Reporte de resultados - Maximización de satisfacción ($z_2$)
---

**Función objetivo $z_1$**

**Pregunta 9 (5 puntos)**

* Guarda en una variable `z1` el valor del costo total de operación y transporte:
>`# Para desarrollo del estudiante`

**Recuerda que** en PuLP puedes usar la función `lp.value(<expresion>)` para evaluar una expresión, reemplazando los valores de las variables por aquellos de la solución óptima. Esta función sólo debe ser llamada luego de usar `<modelo>.solve()` y haber obtenido una solución óptima.

### Fórmula y explicación en palabras — Pregunta 9

**Fórmula (costo total \(z_1\)):**  
\[
z_1 = \sum_{i \in I} f_i\, y_i + \sum_{i,j} c_{ij}\, x_{ij}
\]

**Qué haces en la casilla:** Tras resolver el modelo que **maximiza** \(z_2\), guardas en `z1` el valor de esa expresión usando `lp.value`.

**Por qué:** Ese modelo no minimiza costo; puede ser caro. Necesitas el número para compararlo con el mínimo (`min_costo`) y ver el porcentaje relativo en el reporte.

**Cómo decirlo en voz:** *“Aunque optimicé la satisfacción, mido cuánto me costó en fijos y transporte esa decisión.”*


In [108]:
z1 = lp.value(
    lp.lpSum(costo_fijo[i] * y[i] for i in I)
    + lp.lpSum(costo_transporte[i, j] * x[i, j] for i in I for j in J)
)


In [109]:
print(f"Costo Total: ${z1: .2f}")
print(f"Costo Total Relativo al Mínimo Costo: {z1 / min_costo * 100: .2f}%")

Costo Total: $ 5254425.06
Costo Total Relativo al Mínimo Costo:  121.68%


In [110]:
# Esta celda esta reservada para uso del equipo docente

**Función objetivo $z_2$**

In [111]:
z2 = lp.value(problema.objective)
print(f"Satisfacción Total: {z2: .2f}")
print(
    f"Satisfacción Total Relativa al Total de Producción: {z2 / sum(produccion.values()) * 100: .2f}%"
)

Satisfacción Total:  398.33
Satisfacción Total Relativa al Total de Producción:  94.79%


**Depósitos en operación**

In [112]:
print("Se decidió operar", sum([y[i].value() for i in I]), "depósitos")

Se decidió operar 19.0 depósitos


**Asignación de CACs a Depósitos**

In [113]:
matrix = []
for j in J:
    row = []
    for i in I:
        if y[i].value() == 1:
            if x[i, j].value() == 1:
                row.append("X")
            elif x[i, j].value() == 0:
                row.append("-")
            else:
                row.append("Error")
    matrix.append(row)

df = pd.DataFrame(matrix, index=J, columns=[i for i in I if y[i].value() == 1])
df.head(10)

,"Medellín, Antioquia","La Dorada, Caldas","Aguadas, Caldas","Salamina, Caldas","Popayán, Cauca","Valledupar, Cesar","Bogotá, Cundinamarca","Santana, Huila","Neiva, Huila","Santa Marta, Magdalena","Cúcuta, Nor. de Santander","Pasto, Nariño","Génova, Quindío","Calarcá, Quindío","Filandia, Quindío","Pereira, Risaralda","Bucaramanga, Santander","Barbosa, Santander","Cali, Valle del Cauca"
"Andes, Antioquia",X,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-
"Medellín, Antioquia",-,-,X,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-
"Dabeiba, Antioquia",X,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-
"Salgar, Antioquia",X,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-
"San Pablo de Borbur, Boyacá",-,-,-,-,-,-,X,-,-,-,-,-,-,-,-,-,-,-,-
"Labranzagrande, Boyacá",-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,X,-
"Miraflores, Boyacá",-,-,-,-,-,-,X,-,-,-,-,-,-,-,-,-,-,-,-
"Moniquirá, Boyacá",-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,X,-
"Manizales, Caldas",-,-,-,X,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-
"Anserma, Caldas",-,X,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-


### Visualizaciones
---

**Mapa de la asignación**

In [114]:
m = folium.Map(location=[6.2, -74.5], tiles="OpenStreetMap", zoom_start=6)

for j, lat_lon in cacs_lat_lon.items():
    folium.Marker(
        location=lat_lon,
        tooltip=j,
        icon=BeautifyIcon(
            icon="circle",
            inner_icon_style="color:blue;font-size:7px;opacity:0.9;position: relative;top:-0.5px;",
            background_color="transparent",
            border_color="transparent",
        ),
    ).add_to(m)
for i, lat_lon in depositos_lat_lon.items():
    if y[i].value() > 0:
        folium.Marker(
            location=lat_lon,
            tooltip=i,
            icon=BeautifyIcon(
                icon="caret-up",
                inner_icon_style="color:red;font-size:20px;opacity:0.9;position: relative;top:-4.5px;",
                background_color="transparent",
                border_color="transparent",
            ),
        ).add_to(m)

red = [(i, j) for i in I for j in J if x[i, j].value() > 0]
for i, j in red:
    folium.PolyLine(
        [depositos_lat_lon[i], cacs_lat_lon[j]], color="black", weight=1, opacity=1
    ).add_to(m)

m

## Modelado - Maximización de satisfacción ($z_2$) con restricción de costos ($z_1$)
---
Por último, queremos encontrar un solución intermedia entre la que minimiza los costos y la que maximiza la satisfacción. Hay varias maneras de hacer esto. La que vamos a implementar es colocar una restricción sobre los costos $z_1$ que esté entre los valores obtenidos en los dos casos anteriores.

Recordemos que al minimizar los costos, se obtuvo un costo total de 4,318,336.74. Este costo será nuestro punto de referencia. No podemos obtener un costo menor a este. Por otro lado, al maximizar la satisfacción, obtuvimos un costo de 4,837,569.38. Así que en el peor de los casos, el costo es aproximadamente 12.02% mayor al primer caso. Entonces, para obtener una solución intermedia, debemos escoger un umbral entre estos dos valores para crear una restricción sobre los costos. Una posibilidad es restringir que el costo total $z_1$ sea a lo sumo 2% mayor que el mejor costo mientras se maximiza la satisfacción $z_2$.

### Declaración del modelo

In [115]:
problema = lp.LpProblem(sense=lp.LpMaximize)

### Variables de Decisión

>* $x_{ij}=\begin{cases}1, & \text{si el CAC } j \in J \text{ es atendido por el depósito } i \in I \\0, & \text{de lo contrario} \end{cases}$
>* $y_{i}=\begin{cases} 1, & \text{Si se decide operar el depósito } i \in I  \\ 0, & \text{de lo contrario} \end{cases}$

In [116]:
x = lp.LpVariable.dicts("atender", [(i, j) for i in I for j in J], cat=lp.LpBinary)
y = lp.LpVariable.dicts("operar", I, cat=lp.LpBinary)

### Función Objetivo
Maximizar satisfacción de los CACs
>$\max z_2 = \sum_{j \in J} d_{j} \sum_{\{i \in I | h_{ij} \leq r\}} x_{ij}$

**Pregunta 10 (5 puntos)**
* Crea la función objetivo y agrégala al modelo `problema`

> **Ejemplo**:
>> $ \sum_{i \in I}c_i x_i$
es equivalente a `lp.lpSum(c[i]*x[i] for i in I)`

### Fórmula y explicación en palabras — Pregunta 10

**Fórmula (misma satisfacción \(z_2\)):**  
\[
\max z_2 = \sum_{j \in J}\;\sum_{\{i \in I \mid h_{ij} \leq r\}} d_j\, x_{ij}
\]

**Qué haces en la casilla:** Defines el objetivo del tercer problema: maximizar esa suma (en código, solo incluyes pares con `distancia[(i,j)] <= r`).

**Por qué:** Igual que en la Pregunta 6, pero ahora además vendrá la restricción R3 de costo; el objetivo sigue siendo “máxima producción bien atendida” dentro del presupuesto lógico permitido.

**Cómo decirlo en voz:** *“Sigo queriendo maximizar la producción atendida cerca, pero más adelante limito cuánto puedo gastar en total.”*


In [117]:
problema += lp.lpSum(
    produccion[j] * x[i, j]
    for j in J
    for i in I
    if distancia[(i, j)] <= r
)


In [118]:
# Esta celda esta reservada para uso del equipo docente

In [119]:
# Esta celda esta reservada para uso del equipo docente

### Restricciones
____

**Pregunta 11 (10 puntos)**

* Crea la siguiente restricción, asígnale el nombre `'R1_'+str(<indice_del_para_todo>)` y añádela al modelo:
>1. Cada CAC debe ser atendido por un único depósito
>>`# Para desarrollo del estudiante`

### Fórmula y explicación en palabras — Pregunta 11 (R1, tercer modelo)

**Fórmula:**  
\[
\sum_{i \in I} x_{ij} = 1 \qquad \forall j \in J
\]

**Qué haces en la casilla:** Una ecuación por cada CAC: exactamente un depósito lo atiende; nombres `'R1_'+str(j)`.

**Por qué:** Sin esto un CAC podría quedar sin depósito o con varios, lo cual no tiene sentido en el enunciado.

**Cómo decirlo en voz:** *“A cada centro de acopio le asigno un único depósito responsable.”*


In [120]:
for j in J:
    problema += lp.lpSum(x[i, j] for i in I) == 1, "R1_" + str(j)


In [121]:
# Esta celda esta reservada para uso del equipo docente

**Pregunta 12 (5 puntos)**

* Crea la siguiente restricción, asígnale el nombre `'R2_'+str(<indice_del_para_todo>)` y añádela al modelo:
>2. No se debe superar la capacidad de los depósitos y sólo se puede atender CACs desde un depósito si se decide operar el mismo.
>>$\sum_{j \in J}d_{j}x_{ij} \leq k_{i}y_{i}, \; \forall i \in I$

### Fórmula y explicación en palabras — Pregunta 12 (R2, tercer modelo)

**Fórmula:**  
\[
\sum_{j \in J} d_j\, x_{ij} \;\leq\; k_i\, y_i \qquad \forall i \in I
\]

**Qué haces en la casilla:** Repites la misma restricción de capacidad y de enlace con \(y_i\) que en el modelo anterior, con nombres `'R2_'+str(i)`.

**Por qué:** Las reglas del negocio (capacidad y “solo asigno si el depósito está abierto”) no cambian; solo cambia el objetivo y, en este bloque, la restricción extra de costo.

**Cómo decirlo en voz:** *“Cada depósito no puede recibir más tonelaje del que admite, y solo puedo usar rutas desde depósitos que decida abrir.”*


In [122]:
for i in I:
    problema += (
        lp.lpSum(produccion[j] * x[i, j] for j in J) <= capacidad[i] * y[i]
    ), "R2_" + str(i)


In [123]:
# Esta celda esta reservada para uso del equipo docente

**Pregunta 13 (10 puntos)**

* Crea la siguiente restricción, asígnale el nombre `'R3'` y añádela al modelo:
>El costo total no debe superar en más de un 2% al mejor costo obtenido.
>>`# Para desarrollo del estudiante`

**Nota:** Utiliza el valor `z1_` a continuación como el mejor costo obtenido. Inclúyelo en la restricción según sea conveniente.

In [124]:
z1_ = 4318336.74

### Fórmula y explicación en palabras — Pregunta 13 (restricción R3)

**Fórmula:**  
\[
\sum_{i \in I} f_i\, y_i + \sum_{i,j} c_{ij}\, x_{ij} \;\leq\; 1.02\, z_1^{*}
\]  
donde \(z_1^{*}\) es el valor dado en el notebook como `z1_` (mejor costo del primer modelo).

**Qué haces en la casilla:** Añades al modelo una restricción lineal con nombre `'R3'`: el costo total no puede superar en más del 2 % ese costo de referencia.

**Por qué:** Obligas al optimizador a quedarse en una “banda” de costos alrededor del mínimo, mientras sigue maximizando \(z_2\). Es un compromiso explícito entre dinero y servicio.

**Cómo decirlo en voz:** *“Amarro el costo total para que no pase más de un dos por ciento del mejor costo conocido, y dentro de eso pido la mayor satisfacción posible.”*


In [125]:
problema += (
    lp.lpSum(costo_fijo[i] * y[i] for i in I)
    + lp.lpSum(costo_transporte[i, j] * x[i, j] for i in I for j in J)
    <= z1_ * 1.02
), "R3"


In [126]:
# Esta celda esta reservada para uso del equipo docente

In [127]:
# Esta celda esta reservada para uso del equipo docente

### Invocar el optimizador

In [128]:
print(lp.LpStatus[problema.solve()])

Optimal


## Reporte de resultados - Maximización de satisfacción ($z_2$) con restricción de costos ($z_1$)
---

**Función objetivo $z_1$**

**Pregunta 14 (5 puntos)**

* Guarda en una variable `z1` el valor del costo total de operación y transporte:
>`# Para desarrollo del estudiante`

**Recuerda que** en PuLP puedes usar la función `lp.value(<expresion>)` para evaluar una expresión, reemplazando los valores de las variables por aquellos de la solución óptima. Esta función sólo debe ser llamada luego de usar `<modelo>.solve()` y haber obtenido una solución óptima.

### Fórmula y explicación en palabras — Pregunta 14

**Fórmula (costo total \(z_1\)):**  
\[
z_1 = \sum_{i \in I} f_i\, y_i + \sum_{i \in I}\sum_{j \in J} c_{ij}\, x_{ij}
\]

**Qué haces en la casilla:** Con `lp.value(...)` calculas el valor numérico de esa suma usando la solución **ya obtenida** del tercer modelo (maximizar \(z_2\) con tope de costo).

**Por qué:** En ese modelo el objetivo sigue siendo la satisfacción; el costo no aparece en el objetivo, pero necesitas reportarlo. Así respondes: “¿cuánto costó esta solución intermedia?”.

**Cómo decirlo en voz:** *“Evalúo el costo fijo más el transporte con los valores óptimos de las variables, porque el solver optimizó otra cosa y el costo lo calculo aparte.”*


In [129]:
z1 = lp.value(
    lp.lpSum(costo_fijo[i] * y[i] for i in I)
    + lp.lpSum(costo_transporte[i, j] * x[i, j] for i in I for j in J)
)


In [130]:
print(f"Costo Total: ${z1: .2f}")
print(f"Costo Total Relativo al Mínimo Costo: {z1 / min_costo * 100: .2f}%")

Costo Total: $ 4403765.23
Costo Total Relativo al Mínimo Costo:  101.98%


In [131]:
# Esta celda esta reservada para uso del equipo docente

**Función objetivo $z_2$**

In [132]:
z2 = lp.value(problema.objective)
print(f"Satisfacción Total: {z2: .2f}")
print(f"Satisfacción Total Relativa al Total de Producción: {z2 / sum(produccion.values()) * 100: .2f}%")

Satisfacción Total:  396.67
Satisfacción Total Relativa al Total de Producción:  94.39%


**Depósitos en operación**

In [133]:
print("Se decidió operar", sum(y[i].value() for i in I), "depósitos")

Se decidió operar 17.0 depósitos


**Asignación de CACs a Depósitos**

In [134]:
matrix = []
for j in J:
    row = []
    for i in I:
        if y[i].value() == 1:
            if x[i, j].value() == 1:
                row.append("X")
            elif x[i, j].value() == 0:
                row.append("-")
            else:
                row.append("Error")
    matrix.append(row)

df = pd.DataFrame(matrix, index=J, columns=[i for i in I if y[i].value() == 1])
df.head(10)

,"Medellín, Antioquia","La Dorada, Caldas","Aguadas, Caldas","Salamina, Caldas","Popayán, Cauca","Valledupar, Cesar","Santana, Huila","Neiva, Huila","Santa Marta, Magdalena","Cúcuta, Nor. de Santander","Pasto, Nariño","Génova, Quindío","Calarcá, Quindío","Filandia, Quindío","Bucaramanga, Santander","Barbosa, Santander","Cali, Valle del Cauca"
"Andes, Antioquia",X,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-
"Medellín, Antioquia",X,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-
"Dabeiba, Antioquia",X,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-
"Salgar, Antioquia",X,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-
"San Pablo de Borbur, Boyacá",-,X,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-
"Labranzagrande, Boyacá",-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,X,-
"Miraflores, Boyacá",-,X,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-
"Moniquirá, Boyacá",-,-,-,-,-,-,-,-,-,-,-,-,-,-,-,X,-
"Manizales, Caldas",-,-,-,X,-,-,-,-,-,-,-,-,-,-,-,-,-
"Anserma, Caldas",-,X,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-


### Visualizaciones
---

**Mapa de la asignación**

In [135]:
m = folium.Map(location=[6.2, -74.5], tiles="OpenStreetMap", zoom_start=6)

for j, lat_lon in cacs_lat_lon.items():
    folium.Marker(
        location=lat_lon,
        tooltip=j,
        icon=BeautifyIcon(
            icon="circle",
            inner_icon_style="color:blue;font-size:7px;opacity:0.9;position: relative;top:-0.5px;",
            background_color="transparent",
            border_color="transparent",
        ),
    ).add_to(m)

for i, lat_lon in depositos_lat_lon.items():
    if y[i].value() > 0:
        folium.Marker(
            location=lat_lon,
            tooltip=i,
            icon=BeautifyIcon(
                icon="caret-up",
                inner_icon_style="color:red;font-size:20px;opacity:0.9;position: relative;top:-4.5px;",
                background_color="transparent",
                border_color="transparent",
            ),
        ).add_to(m)

red = [(i, j) for i in I for j in J if x[i, j].value() > 0]
for i, j in red:
    folium.PolyLine(
        [depositos_lat_lon[i], cacs_lat_lon[j]], color="black", weight=1, opacity=1
    ).add_to(m)
m

### Reflexión
---
¿De qué forma podrías obtener soluciones intermedias adicionales? ¿Podrías presentarlas gráficamente como una frontera de Pareto? ¿Si tuvieras que recomendar alguna solución, con qué criterio la escogerías?